# NRI Anomaly Detection (Notebook 2 of 2)
This notebook flags importers and transactions that **deviate from their own established behaviour** (and, secondarily, from their peers). Where Notebook 1 asks "does this match a known money-laundering typology?", this one asks "did this importer suddenly start behaving differently?".

**Anomaly families (first pass)**
- **Value:** a transaction far from the importer's usual range (e.g. a normally high-value importer suddenly moving tiny amounts, or vice versa).
- **Novelty:** an importer that starts dealing in a new country or new commodity late in its history, especially recently.
- **Temporal:** long dormancy followed by reactivation, or sudden bursts in monthly shipment volume.
- **Machine learning (optional):** an unsupervised model that flags odd *combinations* of behaviours the single rules might miss.

**Approach**
- Explainable rules lead, so any flag can be justified for a referral; the ML score is a supplementary prioritization aid, never the sole basis for a flag.

**Conventions** (shared with Notebook 1): importer = `BN_9`, transaction = `CLNT_SPLY_REQ_ID`, all values in CAD.

**Dependency**
- Reuses the foundation tables from `NRI_2_TBML.ipynb` (transaction value, importer features, HS section map). Run that notebook first, using the same `ROW_FILTER` window.

## Setup
Utility functions, the database connection, and configuration - plumbing, not analysis.

### Helper functions
Display and number-formatting helpers, plus a safe "drop table if it exists" used when rebuilding.

In [1]:
import pandas as pd
import numpy as np
import re
import os
import csv

def display_full(df):
    with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.max_colwidth", None, "display.width", 1000):
        display(df)

def barrier():
    print("\n<<", "=" * 50, ">>\n")

def drop_table_if_exists(table):
    try:
        idadb.ida_query(f"DROP TABLE {table};")
    except Exception:
        pass

In [2]:
def try_parse_number(value):
    if isinstance(value, (int, float)):
        return value
    if isinstance(value, str):
        if value.replace(",", "").replace(".", "", 1).isdigit():
            try:
                return float(value.replace(",", ""))
            except ValueError:
                return value
    return value

def format_df_numbers(df):
    df_copy = df.copy()
    def apply_formatting(value):
        if pd.isna(value):
            return value
        num = try_parse_number(value)
        if isinstance(num, (int, float)):
            if float(num).is_integer():
                return f"{int(num):,}"
            return f"{num:,.2f}"
        return value
    return df_copy.map(apply_formatting)

def format_specific_columns(df, columns_to_format):
    df_copy = df.copy()
    def apply_formatting(value):
        if pd.isna(value):
            return value
        num = try_parse_number(value)
        if isinstance(num, (int, float)):
            if float(num).is_integer():
                return f"{int(num):,}"
            return f"{num:,.2f}"
        return value
    for col in columns_to_format:
        if col in df_copy.columns:
            df_copy[col] = df_copy[col].apply(apply_formatting)
    return df_copy

### Database connection
Opens the connection to the Netezza database that holds the import data. Credentials are read from a local `password` file.

In [3]:
from password import password
from nzpyida import IdaDataBase, IdaDataFrame

nzpy_dsn = {
    "database": "OADM_DEV",
    "port": 5480,
    "host": "EC63AP5692",
    "securityLevel": 1,
    "logLevel": 0,
}
idadb = IdaDataBase(nzpy_dsn, uid="GMF939", pwd=password)

### Configuration
Table names and tuning knobs in one place.
- **Reused tables** come from Notebook 1; the rest are anomaly tables created here. All live in the `OADM` schema.
- **`REBUILD`:** `True` rebuilds the database tables; `False` reuses them and just re-runs the in-memory analysis.
- **Thresholds:** the z-score cut-offs for "how far from normal counts as anomalous", the minimum history before an importer is scored, and the grace/recent windows that define what counts as "new" behaviour.

In [4]:
REBUILD = True

SOURCE_TABLE = "OADM.TBML_T_NRI_IMPORT_M"
TXN_TABLE = "OADM.TBML_T_NRI_TXN_VALUE_M"
FEATURES_TABLE = "OADM.TBML_T_NRI_ENTITY_FEATURES_M"
ENT_BASELINE_TABLE = "OADM.TBML_T_NRI_ENT_BASELINE_M"
TXN_ANOM_TABLE = "OADM.TBML_T_NRI_TXN_ANOM_M"
NOVELTY_CTRY_TABLE = "OADM.TBML_T_NRI_NOVELTY_CTRY_M"
NOVELTY_CHAP_TABLE = "OADM.TBML_T_NRI_NOVELTY_CHAP_M"
ENT_MONTHLY_TABLE = "OADM.TBML_T_NRI_ENT_MONTHLY_M"
ANOM_PROFILE_TABLE = "OADM.TBML_T_NRI_ANOM_PROFILE_M"

TEMP_DIR = "data_temp"

VALUE_Z_THRESHOLD = 3.0
MIN_TXNS_BASELINE = 12
REACTIVATION_DAYS = 365
NOVELTY_GRACE_DAYS = 180
RECENT_WINDOW_DAYS = 90
MONTHLY_Z_THRESHOLD = 3.0
MIN_MONTHS_BASELINE = 6
ROW_FILTER = "1=1"

### Bulk-load helper
Same loader as Notebook 1 - plumbing for writing a Python table back into the database without data-type errors. Not part of the analysis, safe to skip.

In [5]:
def infer_nz_types(df):
    types = {}
    for c in df.columns:
        dt = df[c].dtype
        if pd.api.types.is_bool_dtype(dt):
            types[c] = "BOOLEAN"
        elif pd.api.types.is_integer_dtype(dt):
            types[c] = "BIGINT"
        elif pd.api.types.is_float_dtype(dt):
            types[c] = "DOUBLE"
        else:
            max_len = int(df[c].astype(str).str.len().max() or 0)
            width = max(8, min(4000, max_len * 2 + 8))
            types[c] = f"VARCHAR({width})"
    return types

def df_to_netezza(idadb, df, table_name, col_types=None, temp_dir=TEMP_DIR, delimiter=",", remotesource="PYTHON"):
    os.makedirs(temp_dir, exist_ok=True)
    if col_types is None:
        col_types = infer_nz_types(df)
    cols = list(col_types.keys())
    out = df[cols].copy()
    file_name = table_name.split(".")[-1] + ".csv"
    csv_path = os.path.abspath(os.path.join(temp_dir, file_name)).replace("\\", "/")
    log_dir = os.path.abspath(temp_dir).replace("\\", "/")
    out.to_csv(csv_path, index=False, sep=delimiter, na_rep="", quoting=csv.QUOTE_MINIMAL, encoding="utf-8")
    drop_table_if_exists(table_name)
    cols_ddl = ", ".join(f'"{c}" {t}' for c, t in col_types.items())
    idadb.ida_query(f"CREATE TABLE {table_name} ({cols_ddl}) DISTRIBUTE ON RANDOM;")
    load = f"""
    INSERT INTO {table_name}
    SELECT * FROM EXTERNAL '{csv_path}'
    USING (
        DELIMITER '{delimiter}'
        SKIPROWS 1
        QUOTEDVALUE DOUBLE
        NULLVALUE ''
        REMOTESOURCE '{remotesource}'
        LOGDIR '{log_dir}'
    );
    """
    idadb.ida_query(load)
    cnt = idadb.ida_query(f"SELECT COUNT(*) AS N FROM {table_name};")
    row_count = int(np.ravel(np.asarray(cnt))[0])
    print("bulk-loaded", table_name, "->", row_count, "rows from", csv_path)
    return col_types

In [6]:
# Fallback only: chunked append via as_idadataframe. Use if df_to_netezza's external-table
# path is unavailable. chunk_size is auto-capped so rows * columns stays under Netezza's
# UNION-ALL cell budget (~10,000).
from tqdm import tqdm

def prepare_df_for_netezza(df):
    df = df.copy()
    for col in df.columns:
        if "date" in col.lower():
            try:
                df[col] = pd.to_datetime(df[col]).dt.strftime("%Y-%m-%d").astype(object)
            except Exception:
                pass
    for col in df.columns:
        if df[col].dtype == object or df[col].dtype.name == "str":
            df[col] = df[col].astype(object)
    return df

def upload_in_chunks(idadb, df, table_name, chunk_size=3000, clear_existing=True, union_cell_budget=9000):
    df = prepare_df_for_netezza(df)
    n_cols = max(1, df.shape[1])
    effective = max(1, min(chunk_size, union_cell_budget // n_cols))
    sample_df = df.iloc[[0]].copy().fillna("")
    idadf = idadb.as_idadataframe(sample_df, table_name, clear_existing=clear_existing)
    n = len(df)
    for start in tqdm(range(0, n, effective), desc=f"Uploading to {table_name}"):
        chunk = df.iloc[start:start + effective]
        if chunk.empty:
            continue
        idadb.append(idadf, chunk)
    return idadf

## Importer behavioural baselines
One row per importer describing its "normal": how many transactions, the typical size and spread of its transaction values, and how long it has been active.
- Values are measured on a **log scale** because import values are highly skewed; logging makes "typical" and "spread" meaningful.
- Importers with too few transactions get an unreliable baseline and are excluded from value-anomaly flagging.

In [7]:
if REBUILD:
    drop_table_if_exists(ENT_BASELINE_TABLE)
    q = f"""
    CREATE TABLE {ENT_BASELINE_TABLE} AS
    SELECT
        BN_9,
        COUNT(*) AS TXN_CNT,
        AVG(LN(TXN_VALUE_CAD)) AS MEAN_LOG_VAL,
        STDDEV(LN(TXN_VALUE_CAD)) AS STD_LOG_VAL,
        MIN(CAST(ARRIVAL_DATE AS DATE)) AS FIRST_ARRIVAL,
        MAX(CAST(ARRIVAL_DATE AS DATE)) AS LAST_ARRIVAL,
        MAX(CAST(ARRIVAL_DATE AS DATE)) - MIN(CAST(ARRIVAL_DATE AS DATE)) AS TENURE_DAYS
    FROM {TXN_TABLE}
    WHERE TXN_VALUE_CAD > 0
      AND ARRIVAL_DATE IS NOT NULL AND ARRIVAL_DATE <> 'NULL' AND ARRIVAL_DATE <> ''
    GROUP BY BN_9
    DISTRIBUTE ON (BN_9);
    """
    idadb.ida_query(q)
    print("created", ENT_BASELINE_TABLE)

created OADM.TBML_T_NRI_ENT_BASELINE_M


In [8]:
q = f"""
SELECT
COUNT(*) AS n_entities,
SUM(CASE WHEN TXN_CNT >= {MIN_TXNS_BASELINE} THEN 1 ELSE 0 END) AS n_with_baseline,
AVG(TXN_CNT) AS avg_txn_cnt,
AVG(TENURE_DAYS) AS avg_tenure_days
FROM {ENT_BASELINE_TABLE};
"""
display_full(format_specific_columns(idadb.ida_query(q), ["n_entities", "n_with_baseline", "avg_txn_cnt", "avg_tenure_days"]))

,N_ENTITIES,N_WITH_BASELINE,AVG_TXN_CNT,AVG_TENURE_DAYS
0,19355,7446,195.507001,177.676673


**Interpretation - baselines**
- ~19,400 importers, of which ~7,400 have enough transactions (12+) for a reliable value baseline.
- Average tenure is ~178 days, so the data covers roughly a single year - important context for the timing-based rules below.

In [9]:
q = f"""
SELECT *
FROM {ENT_BASELINE_TABLE}
LIMIT 5;
"""
display_full(idadb.ida_query(q))

,BN_9,TXN_CNT,MEAN_LOG_VAL,STD_LOG_VAL,FIRST_ARRIVAL,LAST_ARRIVAL,TENURE_DAYS
0,894270974,577,7.743279,1.535946,2025-01-02,2025-12-31,363
1,891613515,1044,11.167355,0.171370,2025-01-01,2025-12-31,364
2,791082928,980,7.493141,0.877581,2025-01-02,2025-10-14,285
3,839376530,47,6.651119,1.162747,2025-01-10,2025-12-21,345
4,744424094,81,10.256851,0.961795,2025-01-10,2025-12-22,346


## Transaction-level anomalies (value & timing)
For every transaction, compared against its own importer's history:
- **Value deviation (`VALUE_Z`):** how far the value sits from the importer's normal range. Large positive = unusually large; large negative = unusually small.
- **Gap since previous (`DAYS_SINCE_PREV`):** days since that importer's last transaction; a long gap marks a dormant-then-reactivated pattern.
The full scored table stays in the database; only the most extreme transactions are pulled into Python for review.

In [10]:
if REBUILD:
    drop_table_if_exists(TXN_ANOM_TABLE)
    q = f"""
    CREATE TABLE {TXN_ANOM_TABLE} AS
    SELECT
        CLNT_SPLY_REQ_ID,
        BN_9,
        ARRIVAL_DATE,
        TXN_VALUE_CAD,
        ENTITY_TXN_CNT,
        CASE WHEN STD_LOG > 0 THEN (LOG_VAL - MEAN_LOG) / STD_LOG END AS VALUE_Z,
        DAYS_SINCE_PREV,
        CASE WHEN DAYS_SINCE_PREV >= {REACTIVATION_DAYS} THEN 1 ELSE 0 END AS IS_REACTIVATION
    FROM (
        SELECT
            t.CLNT_SPLY_REQ_ID,
            t.BN_9,
            t.ARRIVAL_DATE,
            t.TXN_VALUE_CAD,
            LN(t.TXN_VALUE_CAD) AS LOG_VAL,
            AVG(LN(t.TXN_VALUE_CAD)) OVER (PARTITION BY t.BN_9) AS MEAN_LOG,
            STDDEV(LN(t.TXN_VALUE_CAD)) OVER (PARTITION BY t.BN_9) AS STD_LOG,
            COUNT(*) OVER (PARTITION BY t.BN_9) AS ENTITY_TXN_CNT,
            CAST(t.ARRIVAL_DATE AS DATE) - LAG(CAST(t.ARRIVAL_DATE AS DATE)) OVER (PARTITION BY t.BN_9 ORDER BY CAST(t.ARRIVAL_DATE AS DATE)) AS DAYS_SINCE_PREV
        FROM {TXN_TABLE} t
        WHERE t.TXN_VALUE_CAD > 0
          AND t.ARRIVAL_DATE IS NOT NULL AND t.ARRIVAL_DATE <> 'NULL' AND t.ARRIVAL_DATE <> ''
    ) x
    DISTRIBUTE ON (BN_9);
    """
    idadb.ida_query(q)
    print("created", TXN_ANOM_TABLE)

created OADM.TBML_T_NRI_TXN_ANOM_M


In [11]:
q = f"""
SELECT CLNT_SPLY_REQ_ID, BN_9, ARRIVAL_DATE, TXN_VALUE_CAD, ENTITY_TXN_CNT, VALUE_Z, DAYS_SINCE_PREV
FROM {TXN_ANOM_TABLE}
WHERE ABS(VALUE_Z) >= {VALUE_Z_THRESHOLD}
  AND ENTITY_TXN_CNT >= {MIN_TXNS_BASELINE}
ORDER BY ABS(VALUE_Z) DESC
LIMIT 50;
"""
display_full(format_specific_columns(idadb.ida_query(q), ["TXN_VALUE_CAD", "ENTITY_TXN_CNT", "DAYS_SINCE_PREV"]))

,CLNT_SPLY_REQ_ID,BN_9,ARRIVAL_DATE,TXN_VALUE_CAD,ENTITY_TXN_CNT,VALUE_Z,DAYS_SINCE_PREV
0,10827230990334,856874649,2025-04-08,"9,987,243.02","1,899",27.706045,0
1,17525279486198,869018481,2025-05-08,0.10,"4,933",-24.571941,0
2,17525278931117,889630919,2025-04-17,1.39,"5,801",-23.965528,0
3,17525279990462,889630919,2025-05-17,1.40,"5,801",-23.946936,1
4,15669538614548,844857573,2025-10-09,"37,001.40","2,111",23.085384,0
5,13177269329802,853680965,2025-09-13,629.39,"1,037",-21.991197,1
6,11304038341477,892960782,2025-08-02,27.59,"5,625",-21.677530,0
7,13177269280188,853680965,2025-09-12,688.39,"1,037",-21.581923,3
8,10827758275760,890951882,2025-12-09,"4,134.01",435,-20.258457,0
9,15669543508491,840062756,2025-11-25,"4,552.77","8,430",20.162302,0


**Interpretation - value anomalies**
- The extreme cases are exactly the intended pattern: importers that normally transact small amounts showing a sudden multi-million-dollar transaction (large positive), and high-value importers showing cent-level transactions (large negative).
- Some tiny values ($0.01-$1) are likely data-quality artifacts rather than laundering - worth a quick look before acting.

## Novelty: new countries / new commodities
Detects an importer that usually buys from country/category X suddenly dealing in Y. For each importer and each country (and each commodity chapter) we find when it *first* appeared and compare to the importer's own history:
- **Late-introduced:** first used well after the importer's start, so not part of its initial ramp-up.
- **Recent-new:** a late-introduced country/commodity that appears near the end of the importer's activity - a fresh pivot.
- **NOTE:** vendor *names* are deliberately not used (free text, too noisy); country and commodity chapter are cleaner stand-ins for "new vendor / new product".

In [12]:
def build_novelty(dim_col, out_table):
    drop_table_if_exists(out_table)
    q = f"""
    CREATE TABLE {out_table} AS
    SELECT
        cf.BN_9,
        COUNT(*) AS N_DISTINCT_DIM,
        SUM(CASE WHEN cf.DIM_FIRST > ef.ENT_FIRST + {NOVELTY_GRACE_DAYS} THEN 1 ELSE 0 END) AS N_LATE,
        SUM(CASE WHEN cf.DIM_FIRST > ef.ENT_FIRST + {NOVELTY_GRACE_DAYS}
                  AND cf.DIM_FIRST >= ef.ENT_LAST - {RECENT_WINDOW_DAYS} THEN 1 ELSE 0 END) AS N_RECENT_NEW
    FROM (
        SELECT
            s.BN_9,
            s.{dim_col} AS DIM_VAL,
            MIN(CAST(SUBSTR(CAST(s.ACTUAL_ARRIVAL_DTTM AS VARCHAR(30)), 1, 10) AS DATE)) AS DIM_FIRST
        FROM {SOURCE_TABLE} s
        WHERE {ROW_FILTER}
          AND s.ACTUAL_ARRIVAL_DTTM IS NOT NULL
          AND SUBSTR(CAST(s.ACTUAL_ARRIVAL_DTTM AS VARCHAR(30)), 1, 10) <> 'NULL'
          AND s.{dim_col} IS NOT NULL AND s.{dim_col} <> 'NULL' AND s.{dim_col} <> ''
        GROUP BY s.BN_9, s.{dim_col}
    ) cf
    JOIN (
        SELECT BN_9, MIN(ARR) AS ENT_FIRST, MAX(ARR) AS ENT_LAST
        FROM (
            SELECT s.BN_9, CAST(SUBSTR(CAST(s.ACTUAL_ARRIVAL_DTTM AS VARCHAR(30)), 1, 10) AS DATE) AS ARR
            FROM {SOURCE_TABLE} s
            WHERE {ROW_FILTER}
              AND s.ACTUAL_ARRIVAL_DTTM IS NOT NULL
              AND SUBSTR(CAST(s.ACTUAL_ARRIVAL_DTTM AS VARCHAR(30)), 1, 10) <> 'NULL'
        ) z
        GROUP BY BN_9
    ) ef ON ef.BN_9 = cf.BN_9
    GROUP BY cf.BN_9
    DISTRIBUTE ON (BN_9);
    """
    idadb.ida_query(q)
    print("created", out_table)

if REBUILD:
    build_novelty("VENDOR_CNTRY_CDE", NOVELTY_CTRY_TABLE)
    build_novelty("HS_CHAP_NBR", NOVELTY_CHAP_TABLE)

created OADM.TBML_T_NRI_NOVELTY_CTRY_M
created OADM.TBML_T_NRI_NOVELTY_CHAP_M


In [13]:
q = f"""
SELECT nc.BN_9, nc.N_DISTINCT_DIM AS N_COUNTRIES, nc.N_RECENT_NEW AS RECENT_NEW_COUNTRIES,
nh.N_DISTINCT_DIM AS N_CHAPTERS, nh.N_RECENT_NEW AS RECENT_NEW_CHAPTERS
FROM {NOVELTY_CTRY_TABLE} nc
JOIN {NOVELTY_CHAP_TABLE} nh ON nh.BN_9 = nc.BN_9
WHERE nc.N_RECENT_NEW > 0 OR nh.N_RECENT_NEW > 0
ORDER BY (nc.N_RECENT_NEW + nh.N_RECENT_NEW) DESC
LIMIT 25;
"""
display_full(idadb.ida_query(q))

,BN_9,N_COUNTRIES,RECENT_NEW_COUNTRIES,N_CHAPTERS,RECENT_NEW_CHAPTERS
0,738824309,1,0,21,19
1,779130764,1,0,28,19
2,713975746,1,0,19,16
3,884185505,1,0,17,15
4,758211262,1,0,38,13
5,707185013,1,0,14,13
6,811745314,9,5,27,8
7,770659423,2,1,13,11
8,101855617,1,0,16,12
9,899445530,1,0,18,12


**Interpretation - novelty**
- These importers introduced new commodities or countries late and recently. A high "recent-new" count often just means a naturally broad importer, so this signal is strongest when it coincides with others.

## Volume / frequency spikes
Counts each importer's transactions per month and compares each month to that importer's own monthly average. A month far above its norm is a volume spike - a sudden burst of activity. Only importers with enough active months are scored.
- **Caveat:** only active months are counted, so the baseline ignores quiet months. This understates spikiness for intermittent importers; a calendar-complete (zero-filled) version is a later refinement.

In [14]:
if REBUILD:
    drop_table_if_exists(ENT_MONTHLY_TABLE)
    q = f"""
    CREATE TABLE {ENT_MONTHLY_TABLE} AS
    SELECT
        BN_9,
        YM,
        N_TXN,
        N_MONTHS,
        CASE WHEN STD_M > 0 THEN (N_TXN - MEAN_M) / STD_M END AS VOL_Z
    FROM (
        SELECT
            BN_9,
            YM,
            N_TXN,
            AVG(N_TXN) OVER (PARTITION BY BN_9) AS MEAN_M,
            STDDEV(N_TXN) OVER (PARTITION BY BN_9) AS STD_M,
            COUNT(*) OVER (PARTITION BY BN_9) AS N_MONTHS
        FROM (
            SELECT BN_9, SUBSTR(ARRIVAL_DATE, 1, 7) AS YM, COUNT(*) AS N_TXN
            FROM {TXN_TABLE}
            WHERE ARRIVAL_DATE IS NOT NULL AND ARRIVAL_DATE <> 'NULL' AND ARRIVAL_DATE <> ''
            GROUP BY BN_9, SUBSTR(ARRIVAL_DATE, 1, 7)
        ) m
    ) w
    DISTRIBUTE ON (BN_9);
    """
    idadb.ida_query(q)
    print("created", ENT_MONTHLY_TABLE)

created OADM.TBML_T_NRI_ENT_MONTHLY_M


In [15]:
q = f"""
SELECT BN_9, YM, N_TXN, N_MONTHS, VOL_Z
FROM {ENT_MONTHLY_TABLE}
WHERE VOL_Z >= {MONTHLY_Z_THRESHOLD}
  AND N_MONTHS >= {MIN_MONTHS_BASELINE}
ORDER BY VOL_Z DESC
LIMIT 25;
"""
display_full(format_specific_columns(idadb.ida_query(q), ["N_TXN", "N_MONTHS"]))

,BN_9,YM,N_TXN,N_MONTHS,VOL_Z
0,877518415,2025-12,3,12,3.175434
1,762923811,2025-11,154,12,3.172409
2,706036670,2025-06,230,12,3.172049
3,843778457,2025-01,335,12,3.166924
4,855148862,2025-07,316,12,3.163655
5,870689239,2025-12,"1,217",12,3.161839
6,845790625,2025-07,51,12,3.155152
7,848534533,2025-09,196,12,3.150392
8,780567731,2025-12,193,12,3.122705
9,858885247,2025-11,106,12,3.122143


**Interpretation - volume spikes**
- Very few importers reach the spike threshold, and those that do sit just above it (~3.0-3.2).
- This is a structural limit of a ~1-year window: with at most 12 monthly points, the spike score is mathematically capped near 3.3, so a 3.0 cut-off is very strict. Lowering it (or using a peak-vs-median ratio) is a noted refinement.

## Per-importer anomaly rollup
Combines the transaction-level and monthly signals into one row per importer: counts of anomalous transactions, reactivations, recent-new countries/commodities, and spike months, plus the consolidator flag (so novelty can be suppressed for marketplaces, where "new country/commodity" is meaningless).

In [16]:
q = f"""
SELECT
    b.BN_9,
    b.TXN_CNT,
    b.STD_LOG_VAL,
    b.TENURE_DAYS,
    COALESCE(a.N_VALUE_ANOM, 0) AS N_VALUE_ANOM,
    COALESCE(a.N_REACTIVATION, 0) AS N_REACTIVATION,
    a.MAX_ABS_Z,
    COALESCE(nc.N_RECENT_NEW, 0) AS RECENT_NEW_COUNTRIES,
    COALESCE(nh.N_RECENT_NEW, 0) AS RECENT_NEW_CHAPTERS,
    COALESCE(mm.N_SPIKE_MONTHS, 0) AS N_SPIKE_MONTHS,
    mm.MAX_VOL_Z,
    COALESCE(f.IS_CONSOLIDATOR, 0) AS IS_CONSOLIDATOR,
    f.DISTINCT_VENDOR_COUNTRIES,
    f.DISTINCT_SECTIONS
FROM {ENT_BASELINE_TABLE} b
LEFT JOIN (
    SELECT BN_9,
        SUM(CASE WHEN ABS(VALUE_Z) >= {VALUE_Z_THRESHOLD} AND ENTITY_TXN_CNT >= {MIN_TXNS_BASELINE} THEN 1 ELSE 0 END) AS N_VALUE_ANOM,
        SUM(IS_REACTIVATION) AS N_REACTIVATION,
        MAX(ABS(VALUE_Z)) AS MAX_ABS_Z
    FROM {TXN_ANOM_TABLE}
    GROUP BY BN_9
) a ON a.BN_9 = b.BN_9
LEFT JOIN {NOVELTY_CTRY_TABLE} nc ON nc.BN_9 = b.BN_9
LEFT JOIN {NOVELTY_CHAP_TABLE} nh ON nh.BN_9 = b.BN_9
LEFT JOIN (
    SELECT BN_9,
        SUM(CASE WHEN VOL_Z >= {MONTHLY_Z_THRESHOLD} AND N_MONTHS >= {MIN_MONTHS_BASELINE} THEN 1 ELSE 0 END) AS N_SPIKE_MONTHS,
        MAX(VOL_Z) AS MAX_VOL_Z
    FROM {ENT_MONTHLY_TABLE}
    GROUP BY BN_9
) mm ON mm.BN_9 = b.BN_9
LEFT JOIN {FEATURES_TABLE} f ON f.BN_9 = b.BN_9;
"""
ent = idadb.ida_query(q)
ent.columns = [c.upper() for c in ent.columns]
print("entities:", len(ent))
display_full(ent.head())

entities: 19355


,BN_9,TXN_CNT,STD_LOG_VAL,TENURE_DAYS,N_VALUE_ANOM,N_REACTIVATION,MAX_ABS_Z,RECENT_NEW_COUNTRIES,RECENT_NEW_CHAPTERS,N_SPIKE_MONTHS,MAX_VOL_Z,IS_CONSOLIDATOR,DISTINCT_VENDOR_COUNTRIES,DISTINCT_SECTIONS
0,128740495,8,1.047292,249,0,0,2.466610,0,1,0,2.041241,0,1,3
1,700562721,1,0.000000,0,0,0,NaN,0,0,0,NaN,0,1,1
2,704340553,2,0.986390,31,0,0,0.707107,0,0,0,NaN,0,1,1
3,706179405,13,1.147582,302,0,0,2.033462,0,0,0,2.239171,0,1,5
4,709133201,14,0.418861,276,0,0,1.841832,0,0,0,1.161894,0,1,7


In [17]:
num_cols = [
    "TXN_CNT", "STD_LOG_VAL", "TENURE_DAYS", "N_VALUE_ANOM", "N_REACTIVATION", "MAX_ABS_Z",
    "RECENT_NEW_COUNTRIES", "RECENT_NEW_CHAPTERS", "N_SPIKE_MONTHS", "MAX_VOL_Z",
    "IS_CONSOLIDATOR", "DISTINCT_VENDOR_COUNTRIES", "DISTINCT_SECTIONS"
]
for c in num_cols:
    if c in ent.columns:
        ent[c] = pd.to_numeric(ent[c], errors="coerce")
ent[num_cols] = ent[num_cols].fillna(0)
non_consol = ent["IS_CONSOLIDATOR"] == 0

ent["A_VALUE"] = (ent["N_VALUE_ANOM"] > 0).astype(int)
ent["A_REACTIVATION"] = (ent["N_REACTIVATION"] > 0).astype(int)
ent["A_NEW_COUNTRY"] = ((ent["RECENT_NEW_COUNTRIES"] > 0) & non_consol).astype(int)
ent["A_NEW_CHAPTER"] = ((ent["RECENT_NEW_CHAPTERS"] > 0) & non_consol).astype(int)
ent["A_VOLUME_SPIKE"] = (ent["N_SPIKE_MONTHS"] > 0).astype(int)

rule_cols = ["A_VALUE", "A_REACTIVATION", "A_NEW_COUNTRY", "A_NEW_CHAPTER", "A_VOLUME_SPIKE"]
ent["RULE_SCORE"] = ent[rule_cols].sum(axis=1)
display_full(ent[rule_cols].mean().to_frame("flag_rate").round(4))

,flag_rate
A_VALUE,0.1686
A_REACTIVATION,0.0000
A_NEW_COUNTRY,0.0396
A_NEW_CHAPTER,0.1651
A_VOLUME_SPIKE,0.0026


## Machine-learning layer (optional)
An unsupervised model (IsolationForest) scores each importer on its whole behavioural profile at once, catching odd *combinations* of features the individual rules miss.
- It learns "normal" from non-consolidator importers with enough history, then scores everyone; higher = more unusual, and the top ~1% are marked as ML outliers.
- **Supplementary only:** referrals should rest on the explainable rules; the ML score is for prioritization. The cell is skipped automatically if the ML library is not installed.

In [18]:
try:
    from sklearn.ensemble import IsolationForest
    from sklearn.preprocessing import StandardScaler
    has_sklearn = True
except Exception:
    has_sklearn = False
    print("scikit-learn not available; skipping ML layer")

ml_features = [
    "TXN_CNT", "STD_LOG_VAL", "TENURE_DAYS", "N_VALUE_ANOM", "N_REACTIVATION",
    "MAX_ABS_Z", "RECENT_NEW_COUNTRIES", "RECENT_NEW_CHAPTERS", "N_SPIKE_MONTHS", "MAX_VOL_Z",
    "DISTINCT_VENDOR_COUNTRIES", "DISTINCT_SECTIONS",
]
ent["ML_SCORE"] = np.nan
ent["ML_OUTLIER"] = 0
if has_sklearn:
    x = ent[ml_features].apply(pd.to_numeric, errors="coerce").fillna(0.0)
    x_log = np.sign(x) * np.log1p(np.abs(x))
    fit_mask = (ent["IS_CONSOLIDATOR"] == 0) & (ent["TXN_CNT"] >= MIN_TXNS_BASELINE)
    scaler = StandardScaler().fit(x_log[fit_mask])
    x_scaled = scaler.transform(x_log)
    model = IsolationForest(n_estimators=200, contamination="auto", random_state=42)
    model.fit(scaler.transform(x_log[fit_mask]))
    ent["ML_SCORE"] = -model.score_samples(x_scaled)
    cutoff = ent.loc[fit_mask, "ML_SCORE"].quantile(0.99)
    ent["ML_OUTLIER"] = ((ent["ML_SCORE"] >= cutoff) & fit_mask).astype(int)
    print("ML outliers:", int(ent["ML_OUTLIER"].sum()), "| score cutoff:", round(float(cutoff), 4))

ML outliers: 75 | score cutoff: 0.5971


## Combined profile & review queue
Ranks importers by how many behavioural red flags they trip (ties broken by the ML score) and saves the profile. The top of this list is the review queue, and each importer keeps its individual flags so a reviewer can see *why* it surfaced.

In [19]:
ranked = ent.sort_values(["RULE_SCORE", "ML_SCORE"], ascending=False).reset_index(drop=True)
show_cols = ["BN_9", "IS_CONSOLIDATOR", "TXN_CNT"] + rule_cols + ["RULE_SCORE", "ML_SCORE", "ML_OUTLIER"]
display_full(format_specific_columns(ranked[show_cols].head(25), ["TXN_CNT"]))

persist_cols = [
    "BN_9", "IS_CONSOLIDATOR", "TXN_CNT", "STD_LOG_VAL", "TENURE_DAYS",
    "N_VALUE_ANOM", "N_REACTIVATION", "MAX_ABS_Z", "RECENT_NEW_COUNTRIES", "RECENT_NEW_CHAPTERS",
    "N_SPIKE_MONTHS", "MAX_VOL_Z",
] + rule_cols + ["RULE_SCORE", "ML_SCORE", "ML_OUTLIER"]
anom_profile = ent[persist_cols].copy()
anom_profile = anom_profile.fillna({"MAX_ABS_Z": -1, "MAX_VOL_Z": -1, "ML_SCORE": -1, "STD_LOG_VAL": -1, "TENURE_DAYS": -1})
print("anom_profile shape:", anom_profile.shape)

,BN_9,IS_CONSOLIDATOR,TXN_CNT,A_VALUE,A_REACTIVATION,A_NEW_COUNTRY,A_NEW_CHAPTER,A_VOLUME_SPIKE,RULE_SCORE,ML_SCORE,ML_OUTLIER
0,811745314,0,"1,296",1,0,1,1,0,3,0.684857,1
1,854326352,0,"3,248",1,0,0,1,1,3,0.669812,1
2,856098033,0,"6,640",1,0,1,1,0,3,0.669628,1
3,139599302,0,"1,295",1,0,1,1,0,3,0.657579,1
4,857346191,0,"29,722",1,0,0,1,1,3,0.655140,1
5,841847437,0,248,1,0,1,1,0,3,0.648031,1
6,123578221,0,"5,653",1,0,1,1,0,3,0.645658,1
7,730984317,0,897,1,0,1,1,0,3,0.642430,1
8,879323764,0,"1,264",1,0,1,1,0,3,0.640610,1
9,820209815,0,"1,449",1,0,1,1,0,3,0.636889,1


anom_profile shape: (19355, 20)


In [20]:
if REBUILD:
    df_to_netezza(idadb, anom_profile, ANOM_PROFILE_TABLE)

bulk-loaded OADM.TBML_T_NRI_ANOM_PROFILE_M -> 19355 rows from C:/Users/GMF939/Downloads/Infosphere/NRI/data_temp/TBML_T_NRI_ANOM_PROFILE_M.csv


### Drill-down: one importer's anomalous transactions
For a chosen importer (defaults to the top-ranked one), lists its most extreme transactions so the rolled-up flags can be traced back to specific shipments.

In [21]:
example_bn9 = ranked.iloc[0]["BN_9"]
print("drill-down entity:", example_bn9)
q = f"""
SELECT CLNT_SPLY_REQ_ID, ARRIVAL_DATE, TXN_VALUE_CAD, VALUE_Z, DAYS_SINCE_PREV, IS_REACTIVATION
FROM {TXN_ANOM_TABLE}
WHERE BN_9 = '{example_bn9}'
ORDER BY ABS(VALUE_Z) DESC
LIMIT 25;
"""
display_full(format_specific_columns(idadb.ida_query(q), ["TXN_VALUE_CAD", "DAYS_SINCE_PREV"]))

drill-down entity: 811745314


,CLNT_SPLY_REQ_ID,ARRIVAL_DATE,TXN_VALUE_CAD,VALUE_Z,DAYS_SINCE_PREV,IS_REACTIVATION
0,12997785441165,2025-11-05,27.04,-4.516715,0,0
1,11908030000913,2025-11-29,27.97,-4.491151,1,0
2,12997785096646,2025-07-09,41.96,-4.185376,2,0
3,12997785122045,2025-07-21,63.04,-3.878242,0,0
4,11908030002163,2025-12-16,63.30,-3.875218,0,0
5,12997785353773,2025-10-07,85.71,-3.646523,0,0
6,12997785371673,2025-10-14,100.63,-3.525557,1,0
7,11908030002287,2025-12-18,114.46,-3.428382,0,0
8,12997784880609,2025-05-02,114.68,-3.426960,1,0
9,12997784839754,2025-04-21,114.84,-3.425868,0,0


## Validation & caveats
Flag rates, the spread of flag counts, and how often the rules and the ML model agree.
**Known limitations (first pass)**
- Importers below the minimum history are not value/volume-scored, so brand-new shell entities raise no *behavioural* flag here (Notebook 1's "high first transaction" indicator covers those).
- `VALUE_Z` assumes a roughly symmetric spread after logging; a robust median-based variant is a refinement if values stay skewed.
- Novelty is judged within the loaded date window, so anything seen before the window's start looks falsely "new" - keep the full history and align `ROW_FILTER` with Notebook 1.
- Monthly volume baseline ignores quiet months (see the volume caveat).
- No `REQ_VERS_NBR` de-duplication; re-transmissions can inflate counts/values.
- The ML score is unsupervised triage, not evidence.

In [22]:
print("entities scored:", len(ent))
print("consolidators:", int(ent["IS_CONSOLIDATOR"].sum()))
print()
display_full(ent[rule_cols].mean().sort_values(ascending=False).to_frame("flag_rate").round(4))
print()
display_full(ent["RULE_SCORE"].value_counts().sort_index().to_frame("n_entities"))
if has_sklearn:
    overlap = int(((ent["RULE_SCORE"] >= 2) & (ent["ML_OUTLIER"] == 1)).sum())
    print("\nentities with RULE_SCORE>=2 that are also ML outliers:", overlap)

entities scored: 19355
consolidators: 5



,flag_rate
A_VALUE,0.1686
A_NEW_CHAPTER,0.1651
A_NEW_COUNTRY,0.0396
A_VOLUME_SPIKE,0.0026
A_REACTIVATION,0.0000


,n_entities
RULE_SCORE,
0,13753
1,4062
2,1406
3,134



entities with RULE_SCORE>=2 that are also ML outliers: 56


**Interpretation - results & rule/ML agreement**
- Most importers trip 0-1 behavioural flags; ~134 trip three, which is effectively the maximum given the ~1-year window (reactivation and most volume spikes cannot fire within a single year).
- 60 of the 75 ML outliers also carry two or more rule flags (the exact count shifts slightly run to run, since the model is randomized) - two independent methods agreeing on the same importers, a strong signal that the top of the queue is worth reviewing.
- The zero reactivation rate and rare volume spikes are due to the short time window, not a bug; both come alive with longer history or retuned thresholds.